# Performing a molecular dynamics simulation

In this exercise, we will perform a molecular dynamics simulations of a protein-ligand complex prepared in a previous exercise. It is adapted from [talktorial 19 of TeachOpenCADD](https://github.com/volkamerlab/teachopencadd/tree/master/teachopencadd/talktorials/T019_md_simulation), a platform that aims to teach domain-specific skills and to provide pipeline templates as starting points for research projects.

Assuming that you are reading this Notebook on Github, you will need to download it to your account on SDSC Expanse. To do that, log onto the Expanse User Portal, select *Shell*, and paste the following commands into the terminal:

```bash
mkdir -p ~/exercises
cd ~/exercises
curl -L -o 08-Molecular_Dynamics.ipynb https://raw.githubusercontent.com/daveminh/Chem456-2026S/refs/heads/main/exercises/08-Molecular_Dynamics.ipynb
```

The command to launch a JupyterLab session is similar to previous exercises, but we wil use the `gpu-shared` partition and request 1 Graphical Processing Unit (GPU). GPUs perform individual calculations more slowly than traditional CPUs, but are capable of running a large number of calculations at the same time. Originally developed for generating realistic 3D graphics in computer games, GPUs are now widely used for scientific computation and artificial intelligence.

We will use the `bpsim` conda environment that we set up and used in Exercise 7.
```bash
/cm/shared/apps/sdsc/galyleo/galyleo launch --account iit130 --partition gpu-shared --cpus 4 --gpus 1 --memory 8 --time-limit 02:00:00 --interface lab --conda-env bpsim --conda-init "$HOME/miniconda3/etc/profile.d/conda.sh"
```

The exercise will be graded based on submitting your answers to the questions after ```-->``` on Canvas.

# Part 0 - Setting up the required software

Download [MotorRow](https://github.com/CCBatIIT/MotorRow), a wrapper around OpenMM that **equilibrates a membrane** system. Import packages.

In [ ]:
import os
if not os.path.isdir(os.path.expanduser('~/github/MotorRow')):
    ! mkdir -p ~/github
    %cd ~/github
    ! git clone https://github.com/CCBatIIT/MotorRow.git
    %cd ~/exercises

import sys
sys.path.append(os.path.expanduser('~/github/MotorRow'))
from MotorRow import MotorRow

import numpy as np
import matplotlib.pyplot as plt

import MDAnalysis as mda
from MDAnalysis.analysis import rms, diffusionmap, align
from MDAnalysis import transformations
from MDAnalysis.transformations.translate import center_in_box

import nglview as nv

# Part 1 - Molecular dynamics simulation

## Molecular dynamics

Molecular dynamics is a computational method analyzing the movements and interactions of atoms and molecules of a defined system. The method stems from theoretical physics, where it was developed in the 1950s (Alder and Wainwright in [_J Chem Phys_ (1959), **31**(2), 459](https://doi.org/10.1063/1.1730376)), although the ideas behind it can be dated much earlier:

> An intelligence which could, at  any moment, comprehend all the forces by  which nature is animated and the  respective positions of the  beings of which it is  composed, and moreover, if this intelligence were far-reaching enough to subject these data to  analysis, it would encompass in that formula both the movements of the  largest bodies in  the universe and those of the lightest atom: to it nothing would be uncertain, and the  future, as well as the past, would be present to its eyes. The human mind offers us, in the perfection which it has  given to  astronomy, a faint sketch of this intelligence. (Pierre-Simon Laplace, 1820)


Let us just take this statement by Laplace as the ideological substrate underneath molecular dynamics simulations. In other terms, we can approximate the behavior of a physical system by knowing the characteristics of its components and applying Newton's laws of motion. By solving the equations of motion, we can obtain a molecular trajectory of the system, which is a series of snapshots with the positions and velocities of all its particles, as well as its potential energy. To do so, we define functions, called force fields, which provide an approximate description of all the forces applied to each particle in the system. We then use numerical integrators to solve the initial value problem for the system and obtain the trajectory. As it sounds, the process requires quite a bit of processing power and it was only few years ago that MD started seeing a more widespread use, especially in the field of computational chemistry and biology, as well as in drug discovery ([_J Med Chem_ (2016), **59**(9), 4035‐4061](https://doi.org/10.1021/acs.jmedchem.5b01684)).

![MD_rotor_250K_1ns.gif](https://github.com/volkamerlab/teachopencadd/raw/d1ded86bb2c82ef088cc5145d0bcb997f6eab7dd/teachopencadd/talktorials/018_md_simulation/images/MD_rotor_250K_1ns.gif)

**Figure 1**: Molecular dynamics simulation of the rotation of a supramolecule composed of three molecules in a confined nanoscopic pore (Palma et al. via [Wikimedia](https://commons.wikimedia.org/w/index.php?curid=34866205)).

### MD simulations and drug design

MD simulations give valuable insights into the highly dynamic process of ligand binding to their target. When a ligand (or a drug) approaches a macromolecule (protein) in solution, it encounters a structure in constant motion. Also, ligands may induce conformational changes in the macromolecule that can best accommodate the small molecule. Such conformations may not be discovered with static methods. Accordingly, binding sites that are not observed in static ligand-free structures, but can be discovered with MD simulations, are sometimes called *cryptic binding sites* ([_J Med Chem_ (2016), **59**(9), 4035‐4061](https://doi.org/10.1021/acs.jmedchem.5b01684)). The identification of such binding sites with MD simulation can kickstart new drug discovery campaigns. Later in the drug discovery process, MD simulations can also be used to estimate the quality of computationally identified small molecules before performing more costly and time-intensive *in vitro* tests. Altogether, MD simulations pose a valuable asset in computational drug design.

We will now proceed to perform an MD simulation using the molecular dynamics engine [OpenMM](https://github.com/openmm/openmm), a high performance toolkit for molecular simulation. It is open source and can be used as application or library. We will use it as Python library.

## Loading the system

In Exercise 7, we created an [OpenMM System](http://docs.openmm.org/development/api-python/generated/openmm.openmm.System.html#openmm.openmm.System) and set up the simulation. 

Here we will create a MotorRow instance that loads the system. MotorRow can be run with a single command, but I'll break down the steps so that you can think about them.

Modify the variable `system_prefix` to match one of the systems that you prepared in Exercise 7.

In [ ]:
base_dir = os.path.expanduser('~/exercises/')
system_prefix = 'A1H3H'
state_fn = os.path.join(base_dir, '07', 'systems', system_prefix + '.xml')
pdb_fn = os.path.join(base_dir, '07', 'systems', system_prefix + '.pdb')
ref_pdb_fn = pdb_fn
openmm_dir = os.path.join(base_dir, '08', system_prefix)

if not os.path.isfile(state_fn):
  raise ValueError(f'Input file {state_fn} not found!')
if not os.path.isfile(pdb_fn):
  raise ValueError(f'Input file {pdb_fn} not found!')
if not os.path.isdir(openmm_dir):
  ! mkdir -p {openmm_dir}

MR = MotorRow(pdb_fn, state_fn, openmm_dir)

## Energy minimization
While everything is set up, we need to minimize the energy of the system to get a low energy starting configuration. This is important to decrease the chance of simulation failures due to severe atom clashes. The energy minimized system is saved.

In [ ]:
if not (os.path.isfile(os.path.join(openmm_dir, 'minimized.pdb')) and os.path.isfile(os.path.join(openmm_dir, 'minimized.xml'))):
    state_fn, pdb_fn = MR._minimize(ref_pdb_fn)

#### --> How much does minimization decrease the energy of the system? What is the original energy, final energy, and the difference?

Now visualize the minimized structure.

In [ ]:
view = nv.NGLWidget()
view.add_component(os.path.join(openmm_dir, 'minimized.pdb'))
view.add_representation("ball+stick", selection="water")
view.display()

The minimized system may look strange with a split membrane and the protein protruding into the air. This is an artifact of the periodic boundary conditions. For simpler visualization and analysis, it could help to wrap the system so that the protein and membrane appear to be in the same unit cell.

In [ ]:
view = nv.NGLWidget()
view.add_component(os.path.join(openmm_dir, 'minimized_wrapped.pdb'))
view.add_representation("ball+stick", selection="water")
view.display()

Now let's compare the original and minimized structures. Because OpenMM moves the system, we use mdanalysis to align the frames. Note that nglview allows you to select specific trajectory frames and loop a movie.

In [ ]:
u = mda.Universe(ref_pdb_fn, \
    [ref_pdb_fn, os.path.join(openmm_dir, 'minimized_wrapped.pdb')], in_memory=True)
align.AlignTraj(u, u, select='name CA').run()

view = nv.show_mdanalysis(u)
view.add_representation("ball+stick", selection="water")
view.display()

#### --> Describe what happens to the system during the minimization process

## Equilibration

Once the minimization has finished, we can start equilibration. Equilibration is needed because the initial minimized configuration of the system (suitable for T=0 K) is not representative of a typical configuration in the thermodynamic state. Throwing out an equilibration period reduces bias in estimates of expectation values. In this lab, we will perform an equilibration process suitable for membrane proteins. The results are saved in an .dcd file, which contains the coordinates of all the atoms at a given time point.

### Step 1: NVT with restraints

This step is in the NVT ensemble; there are a constant number of particles, volume, and temperature. There are heavy restraints on the protein and on the Z coordinates of the membrane, allowing the water to equilibrate. The length of Step 1 is 125000 steps x 2 fs/step = 250000 fs = 250 ps.

In [ ]:
if not (os.path.isfile(os.path.join(openmm_dir, 'Step_1.pdb')) and os.path.isfile(os.path.join(openmm_dir, 'Step_1.xml'))):
    state_fn, pdb_fn = MR._run_step(state_fn, 1, nsteps=125000, positions_from_pdb=os.path.join(openmm_dir, 'minimized.pdb'))

For the next few questions, consult the OpenMM [user guide](http://docs.openmm.org/latest/userguide/theory/04_integrators.html) and [documentation](http://docs.openmm.org/latest/api-python/library.html#integrators) about integrators.

#### --> MotorRow uses the LangevinIntegrator from OpenMM. Is the LangevinIntegrator stochastic or deterministic?

#### --> LangevinIntegrator accepts the temperature as an argument. Why?

#### --> Does every integrator accept temperature as an argument? Why or why not?

#### --> With the computer resources that you used, how many nanoseconds of simulation you would be able to perform in a day?

Note that the system is assigned random velocities at the beginning of Step 1.

#### --> How does the potential energy of the system compare after minimization and near the beginning of Step 1? How does it change over the course of Step 1? Why are these trends observed?

In [ ]:
def show_rmsd(traj_fn):
    u = mda.Universe(ref_pdb_fn, traj_fn)   
    R = rms.RMSD(u.select_atoms("backbone"), u.select_atoms("backbone"))
    R.run()
    plt.plot(R.results.rmsd[:,2])
    plt.xlabel('Time step')
    plt.ylabel('RMSD (A)')
show_rmsd(os.path.join(openmm_dir, 'step1.dcd'))

#### --> Why does the RMSD compared to the initial frame quickly jump before changing more gradually?

In [ ]:
def show_traj(traj_fn):
    u = mda.Universe(ref_pdb_fn, traj_fn)
    # Only keep ~50 frames
    total_frames = len(u.trajectory)
    stride = max(1, total_frames // 50)
    u.transfer_to_memory(step=stride)
    
    transform_1 = center_in_box(u.select_atoms("protein")) # Put the protein in the center
    transform_2 = transformations.wrap(u.select_atoms("not protein")) # Put in the box
    transform_3 = transformations.unwrap(u.select_atoms("(not protein) and (not water)")) # Prevent molecules from spanning the box
    u.trajectory.add_transformations(*[transform_1, transform_2, transform_3])
    
    view = nv.show_mdanalysis(u)
    view.add_representation("ball+stick", selection="water and not hydrogen")
    return view

view = show_traj(os.path.join(openmm_dir, 'step1.dcd'))
view.display()

#### --> How does Step 1 affect the density of water near the protein? Hint: you may want to compare the very first and last frame.

### Step 2 NPT with restraints

This step is in the NPT ensemble; there are a constant number of particles, pressure, and temperature. It uses the same restraints as in Step 1. The key difference is that the pressure is equilibrated with a barostat, which changes the box volume. The [membrane barostat](http://docs.openmm.org/7.0.0/api-python/generated/simtk.openmm.openmm.MonteCarloMembraneBarostat.html) was designed such that box size in the Z dimension varies independently from other axes. The length of Step 1 is 125000 steps x 2 fs/step = 250000 fs = 250 ps.

In [ ]:
if not (os.path.isfile(os.path.join(openmm_dir, 'Step_2.pdb')) and os.path.isfile(os.path.join(openmm_dir, 'Step_2.xml'))):
    state_fn, pdb_fn = MR._run_step(state_fn, 2, nsteps=125000, positions_from_pdb=os.path.join(openmm_dir, 'Step_1.pdb'))

In [ ]:
view = show_traj(os.path.join(openmm_dir, 'step2.dcd'))
view.display()

In [ ]:
# Box dimensions as function of time step
def show_box_dimensions(traj_fn):
    u = mda.Universe(ref_pdb_fn, traj_fn, in_memory=True)
    box_dims = np.array([ts.dimensions[:3].copy() for ts in u.trajectory])  
   
    plt.plot(box_dims)
    plt.legend(['x','y','z'], bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xlabel('Time step')
    plt.ylabel('Box dimension (A)')
    
    print(f"Initial box dimensions: {box_dims[0]}")
    print(f"Final box dimensions:   {box_dims[-1]}")
    
show_box_dimensions(os.path.join(openmm_dir, 'step2.dcd'))

#### --> Does the box size change? If so, in which dimensions is the change the largest?

In [ ]:
show_rmsd(os.path.join(openmm_dir, 'step2.dcd'))

### Step 3: NVT with no restraints

Now the restraints are released, allowing the membrane and protein to relax.

In [ ]:
if not (os.path.isfile(os.path.join(openmm_dir, 'Step_3.pdb')) and os.path.isfile(os.path.join(openmm_dir, 'Step_3.xml'))):
    state_fn, pdb_fn = MR._run_step(state_fn, 3, nsteps=250000, positions_from_pdb=os.path.join(openmm_dir, 'Step_2.pdb'))

In [ ]:
view = show_traj(os.path.join(openmm_dir, 'step3.dcd'))
view.display()

#### --> Describe what happens to the membrane in Step 3.

In [ ]:
show_rmsd(os.path.join(openmm_dir, 'step3.dcd'))

### Step 4: NPT without restraints

Now the system is simulated without restraints and with a membrane barostat.

In [ ]:
if not (os.path.isfile(os.path.join(openmm_dir, 'Step_4.pdb')) and os.path.isfile(os.path.join(openmm_dir, 'Step_4.xml'))):
    state_fn, pdb_fn = MR._run_step(state_fn, 4, nsteps=1250000, positions_from_pdb=os.path.join(openmm_dir, 'Step_3.pdb'))

In [ ]:
view = show_traj(os.path.join(openmm_dir, 'step4.dcd'))
view.display()

#### --> Describe what happens to the membrane in Step 4.

In [ ]:
show_box_dimensions(os.path.join(openmm_dir, 'step4.dcd'))

#### --> Does the box size change? If so, in which dimensions is the change the largest?

In [ ]:
show_rmsd(os.path.join(openmm_dir, 'step4.dcd'))

### Step 5: NPT with an isotropic barostat

In [ ]:
if not (os.path.isfile(os.path.join(openmm_dir, 'Step_5.pdb')) and os.path.isfile(os.path.join(openmm_dir, 'Step_5.xml'))):
    state_fn, pdb_fn = MR._run_step(state_fn, 5, nsteps=1250000, positions_from_pdb=os.path.join(openmm_dir, 'Step_4.pdb'))

In [ ]:
view = show_traj(os.path.join(openmm_dir, 'step5.dcd'))
view.display()

In [ ]:
show_box_dimensions(os.path.join(openmm_dir, 'step5.dcd'))

#### --> Does the box size change? If so, in which dimensions is the change the largest?|

In [ ]:
show_rmsd(os.path.join(openmm_dir, 'step5.dcd'))

# Part 2 - A molecular dynamics batch job

We have run MotorRow interactively for one of the systems that you prepared in Exercise 7. Now edit the `system_prefix` variable and run the cell below to run MotorRow with a batch job for the other system. Editing `email_address` will inform you when the job has started and completed. After the job has started, you can shut down JupyterLab. When it has completed, run `tail ~/exercises/08/{system_prefix}/step5.stdout` (replacing {system_prefix}) in the Shell on SDSC Expanse.

#### --> What is the output of the `tail ~/exercises/08/{system_prefix}/step5.stdout`?

In [49]:
email_address = ''
system_prefix = 'MSX-2'

if email_address != '':
    email_lines = f"""#SBATCH --mail-user={email_address}
#SBATCH --mail-type=ALL"""

state_fn = os.path.join(base_dir, '07', 'systems', system_prefix + '.xml')
pdb_fn = os.path.join(base_dir, '07', 'systems', system_prefix + '.pdb')
openmm_dir = os.path.join(base_dir, '08', system_prefix)

if not os.path.isdir(openmm_dir):
  ! mkdir -p {openmm_dir}

with open(f"08/md-{system_prefix}.py","w") as f:
    f.write(f"""import os, sys
sys.path.append(os.path.expanduser('~/github/MotorRow'))
from MotorRow import MotorRow

MR = MotorRow("{pdb_fn}", "{state_fn}", "{openmm_dir}")
MR.main("{pdb_fn}")
""")

with open(f"08/md-{system_prefix}.job","w") as f:
    f.write(f"""#!/bin/bash
#SBATCH --account=iit130
#SBATCH --partition=gpu-shared
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=4
#SBATCH --gpus 1
#SBATCH --mem=10G
#SBATCH --time=04:00:00
#SBATCH --job-name="MotorRow"
#SBATCH --output="MotorRow.%j.%N.out"
#SBATCH --export=ALL
{email_lines}

source "$HOME/miniconda3/etc/profile.d/conda.sh"
conda activate bpsim
cd ~/exercises/08/
python md-{system_prefix}.py
""")

%cd 08
! sbatch md-{system_prefix}.job
%cd ..

/home/daveminh/exercises/08
Submitted batch job 47612748
/home/daveminh/exercises


## Discussion

We have successfully performed an MD simulation of a protein ligand complex in membrane. However, we simulated only a short time to keep the execution time of the lab short. To address critical questions in drug design, longer simulations are often required. Conventional MD simulations are usually too computationally costly to be useful for this purpose. Thus, so-called enhanced sampling methods that aim to accelerate the conformational sampling were developed. We will do this in future exercises.

## Further reading

### Enhanced sampling methods

In theory, unbiased MD simulations should be capable of simulating binding and unbinding events of a drug molecule and its macromolecular target. However, the timescale of binding and unbinding events lies in the millisecond to second range. Enhanced sampling methods aim to accelerate the conformational sampling ([_J Med Chem._ 2016, **59(9)**, 4035-61](https://doi.org/10.1021/acs.jmedchem.5b01684)).

One of these is **Free energy perturbation (FEP)** (also called alchemical free energy calculation), which computes the free energy difference when going from a state A to another state B. It is often employed in lead optimization to evaluate small modification at the ligand, that may boost the binding affinity for the desired target. The ligand from state A is thereby gradually transformed into the ligand of state B by simulating several intermediate ("alchemical") states ([alchemistry](http://www.alchemistry.org/wiki/Main_Page)).

Another technique for free-energy calculations is **Umbrella sampling (US)**. US enforces sampling along a collective variable (CV) by performing staged simulations with an energetic bias. The bias usually takes the form of a harmonic potential, hence the term "umbrella". Its goal is to sample high-energy regions along the CV. However, the use in drug design is limited by the high computational cost.

In contrast, **Steered MD (SMD)** follows a different approach: it applies external forces to the system. Those forces are time-dependent and facilitate the unbinding of the ligand from the target. The SMD calculates the final force exerted on the system. The unbinding force profile can then be used to filter hits from docking calculations and to discriminate active from inactive molecules.

### References

- Review on the impact of MD simulations in drug discovery ([_J Med Chem_ (2016), **59**(9), 4035‐4061](https://doi.org/10.1021/acs.jmedchem.5b01684))
- Review on the physics behind MD simulations and best practices ([_Living J Comp Mol Sci_ (2019), **1**(1), 5957](https://doi.org/10.33011/livecoms.1.1.5957))
- Review on force fields ([_J Chem Inf Model_ (2018), **58**(3), 565-578](https://doi.org/10.1021/acs.jcim.8b00042))
- Review on EGFR in cancer ([_Cancers (Basel)_ (2017), **9**(5), 52](https://dx.doi.org/10.3390%2Fcancers9050052))
- Summarized statistical knowledge from Pierre-Simon Laplace ([Théorie Analytique des Probabilités _Gauthier-Villars_ (1820), **3**)](https://archive.org/details/uvrescompltesde31fragoog/page/n15/mode/2up)
- Inspired by a notebook form Jaime Rodríguez-Guerra ([github](https://github.com/jaimergp/uab-msc-bioinf/blob/master/MD%20Simulation%20and%20Analysis%20in%20a%20Notebook.ipynb))
- Repositories of [OpenMM](https://github.com/openmm/openmm) and [OpenMM Forcefields](https://github.com/openmm/openmmforcefields), [RDKit](https://github.com/rdkit/rdkit), [PyPDB](https://github.com/williamgilpin/pypdb), [MDTraj](https://github.com/mdtraj/mdtraj), [PDBFixer](https://github.com/openmm/pdbfixer)
- Wikipedia articles about [MD simulations](https://en.wikipedia.org/wiki/Molecular_dynamics), [AMBER](https://en.wikipedia.org/wiki/AMBER) and [force fields](https://en.wikipedia.org/wiki/Force_field_(chemistry)) in general